<a href="https://colab.research.google.com/github/vcellmike/PatternsFormation/blob/main/Working/visualizations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import torch
from sklearn.metrics import accuracy_score, classification_report
from PIL import Image
import pandas as pd
from google.colab import drive
from PIL import Image
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
pd.set_option('display.max_columns', None)

In [ ]:
# set random seeds for repeatability
import numpy as np
import random

def set_seed(seed_val):
    random.seed(seed_val)
    np.random.seed(seed_val)
    torch.manual_seed(seed_val)
    torch.cuda.manual_seed_all(seed_val)
seed_val = 42
set_seed(seed_val)

In [ ]:
feats_df = pd.read_pickle("2025_feats_df_.pkl")

In [ ]:
# plot the model predicted class in feature space
def plotter(feat_1,feat_2):
  num_classes = 7
  classes = np.arange(0,num_classes)# list of unique clusters

  class_dfs = [] # list of dfs for each cluster

  for i in classes:
    df = feats_df[feats_df["classifier_pred_class"] == i]
    df.reset_index(inplace = True, drop = True)
    class_dfs.append(df)

  fig, ax = plt.subplots()

  colors = ["red","green","blue","purple","orange","black","pink"]
  alpha = [1,1,1,1,1,1,1]
  for j in classes:
    df = class_dfs[j]
    ax.scatter(df[feat_1], df[feat_2], c = colors[j], label = "class " + str(j), alpha = alpha[j])


  leg = ax.legend(loc="upper left")

  ax.set(xlabel = feat_1, ylabel = feat_2)
  plt.show()

  return

params = ["Ua","Ui","Ga","Gi","Da","Di","Ba"]
"""features = [
    "Circ._std",
    "Round_mean",
    "Area_inverted_mean",
    "Circ._mean",
    "Perim._mean",
    "Perim._inverted_mean",
    "Area_inverted_std",
    "IntDen_inverted_std",
    "RawIntDen_inverted_std"
]"""

for p1 in params:
  for p2 in params:
    if p1 != p2:
      plotter(p1,p2)
      print(p1, " x ", p2)

In [ ]:
# make all of the params type float

feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]] = feats_df[["Ua","Ui","Ga","Gi","Da","Di","Ba"]].astype(float)

# do clusters in first 3 PCs

from mpl_toolkits import mplot3d

def threeD_plotter(feat_1,feat_2,feat_3, elevation=30, azi=90):
    num_classes = 7
    classes = np.arange(0,num_classes)# list of unique clusters

    class_dfs = [] # list of dfs for each cluster

    for i in classes:
      df = feats_df[feats_df["classifier_pred_class"] == i]
      df.reset_index(inplace = True, drop = True)
      class_dfs.append(df)


    # feat_1 = "pc1_imagej_features"

    # feat_2 = "pc2_imagej_features"

    # feat_3 = "pc3_imagej_features"

    # for k in range(10):
    # Creating 3D figure
    fig = plt.figure(figsize=(8, 8))
    ax = plt.axes(projection='3d')

    colors = ["red","green","blue","purple","orange","black","pink"]
    alpha = [1,1,1,1,1,1,0]
    for j in classes:
      df = class_dfs[j]
      ax.scatter3D(df[feat_1], df[feat_2], df[feat_3], c = colors[j], label = "class " + str(j), alpha = alpha[j])


    leg = ax.legend(loc="upper left")

    ax.set(xlabel = feat_1, ylabel = feat_2, zlabel = feat_3)

    # 360 Degree view
    ax.view_init(elevation, azi)
    plt.show()


params = ["Ua","Ui","Ga","Gi","Da","Di","Ba"]

for i in range(len(params)):
  for j in range(i,len(params)):
    for k in range(j,len(params)):
      if i != j and j != k:
        threeD_plotter(params[i],params[j],params[k])

### ui, gi, ga
###for azi_value in range(0,360, 10):
###  threeD_plotter("Ui","Gi","Ga", azi = azi_value)

In [ ]:
## 00A
# plot ratios of params

def ratio_plotter(feat_1,feat_2):
  num_classes = 7
  classes = np.arange(0,num_classes)# list of unique clusters

  class_dfs = [] # list of dfs for each cluster

  for i in classes:
    df = feats_df[feats_df["classifier_pred_class"] == i]
    df.reset_index(inplace = True, drop = True)
    class_dfs.append(df)

  fig, ax = plt.subplots()

  colors = ["red","green","blue","purple","orange","black","pink"]
  alpha = [1,1,1,1,1,1,0]
  for j in classes:
    df = class_dfs[j]
    ax.scatter(df[feat_1], df[feat_2], c = colors[j], label = "class " + str(j), alpha = alpha[j])


  leg = ax.legend(loc="upper left")

  ax.set(xlabel = feat_1, ylabel = feat_2)
  plt.show()

  return


params = ["Ua","Ui","Ga","Gi","Da","Di","Ba"]
counter = 0
for i in range(len(params)):
  for j in range(i,len(params)):
    for k in range(len(params)):
      for l in range(k,len(params)):
        p0 = params[i]
        p1 = params[j]
        p2 = params[k]
        p3 = params[l]

        if p0 != p1 and p1 != p2 and p2 != p3 and p1 != p3 and p0 != p3:
          ## Dumps the feats_df[p0,p1,p2,p3] to numpy arrays and then divides them
          ratio_1 = np.array(feats_df[p0]).astype(float) / np.array(feats_df[p1]).astype(float)
          feats_df[p0 + "/" + p1] = ratio_1

          ratio_2 = np.array(feats_df[p2]).astype(float) / np.array(feats_df[p3]).astype(float)
          feats_df[p2 + "/" + p3] = ratio_2

          ratio_plotter(p0 + "/" + p1, p2 + "/" + p3)
          counter += 1
print(counter)

print("Complete.")

In [ ]:
feats_df.head()

In [ ]:
## THIS CODE BLOCK IS DEPENDANT ON CODE BLOCK MARKED 00A
## This code creates a dictionary of all the features' attributes/properties. Each row (eg. 0 is a collection of values for a specific image)

# Defines features + params + params ratios
feats = ["Ua","Ui","Ga","Gi","Da","Di","Ba",'Ua/Ui', 'Ua/Ga', 'Ua/Gi', 'Ua/Da', 'Ua/Di', 'Ua/Ba', 'Ga/Gi', 'Ga/Da',
       'Ga/Di', 'Ga/Ba', 'Gi/Da', 'Gi/Di', 'Gi/Ba', 'Da/Di', 'Da/Ba', 'Di/Ba',
       'Ui/Gi', 'Ui/Da', 'Ui/Di', 'Ui/Ba']  + list(feats_df.columns[13:111])

feats_extra = []

# creates a list of parameter properties (eg. Ua/Ui Mean, Min, Max, std)
for i in feats:
  feats_extra.append(i + " mean")
  feats_extra.append(i + " min")
  feats_extra.append(i + " max")
  feats_extra.append(i + " std")

feats = feats_extra
#print(len(feats))

num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster
rows = {} ##??? what is this doing?

for i in classes:
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)
  rows[str(i)] = []

print(rows)

for i in range(num_classes):
  df = class_dfs[i]
  for j in range(0,len(feats),4):
    mean_val = np.mean(np.array(df[feats[j][0:feats[j].index(" ")]]).astype(float))
    rows[str(i)].append(mean_val)

    min_val = np.min(np.array(df[feats[j][0:feats[j].index(" ")]].astype(float)))
    rows[str(i)].append(min_val)

    max_val = np.max(np.array(df[feats[j][0:feats[j].index(" ")]].astype(float)))
    rows[str(i)].append(max_val)

    std = np.std(np.array(df[feats[j][0:feats[j].index(" ")]].astype(float)))
    rows[str(i)].append(std)

print(rows)
    # print("Cluster " + str(i) + " avg " + feat + ": " + str(mean_val))

In [ ]:
cluster_means_df = pd.DataFrame(rows)
# cluster_means_df["feats"] = feats
print(cluster_means_df)
cluster_means_df = cluster_means_df.transpose()
print("transpose complete:")
cluster_means_df.columns = feats

cluster_means_df

In [ ]:
feats = ["Ua","Ui","Ga","Gi","Da","Di","Ba",'Ua/Ui', 'Ua/Ga', 'Ua/Gi', 'Ua/Da', 'Ua/Di', 'Ua/Ba', 'Ga/Gi', 'Ga/Da',
       'Ga/Di', 'Ga/Ba', 'Gi/Da', 'Gi/Di', 'Gi/Ba', 'Da/Di', 'Da/Ba', 'Di/Ba',
       'Ui/Gi', 'Ui/Da', 'Ui/Di', 'Ui/Ba'] + list(feats_df.columns[13:111])

num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster
rows = {}

for i in classes:
   df = feats_df[feats_df["classifier_pred_class"] == i]
   df.reset_index(inplace = True, drop = True)
   class_dfs.append(df)
   rows[str(i)] = []

for feature in feats:
  list_of_lists = []
  print(feature)
  for i in range(len(class_dfs)):
    #print(f"We are in class {i}")
    values = np.array(class_dfs[i][feature]).astype(float)
    #print(values)
    list_of_lists.append(list(values))
  plt.boxplot(list_of_lists,positions=classes)
  plt.title(feature + " Broken Down by Class")
  plt.xlabel("Class")
  plt.ylabel(feature)
  plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler,MaxAbsScaler

## 1 box with 7 params on the same graph

num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters
#params = [["Ua","Ui","Ga","Gi","Da","Di","Ba",'Ua/Ui', 'Ua/Ga', 'Ua/Gi', 'Ua/Da', 'Ua/Di', 'Ua/Ba', 'Ga/Gi', 'Ga/Da',
#       'Ga/Di', 'Ga/Ba', 'Gi/Da', 'Gi/Di', 'Gi/Ba', 'Da/Di', 'Da/Ba', 'Di/Ba',
#       'Ui/Gi', 'Ui/Da', 'Ui/Di', 'Ui/Ba'] + (feats_df.columns[13:111])]
params_shortened = ["Ua","Ui","Ga","Gi","Da","Di","Ba"]
key = params_shortened
num_values = len(key) + 1

sc = StandardScaler()

## Loop through classes
for i in classes:
  listoflists = []
  df = feats_df[feats_df["classifier_pred_class"] == i]
  classification = i
  for param in key:
    # Creates a numpy array of all values for the parameters
    print(param)
    numbers = np.array(df[param]).astype(float)
    # scale the data
    numbers = sc.fit_transform(numbers.reshape(-1,1)).flatten()
    numbers.reshape(1,-1)
    #print(numbers)
    # np.append(array, values)
    listoflists.append(numbers)
    #print(numbers)
  plt.boxplot(
      listoflists,
      positions=np.arange(0,num_values - 1).tolist(),
      showfliers=False,
      labels=["Ua","Ui","Ga","Gi","Da","Di","Ba"]
  )
  plt.title("Class " + str(classification))
  plt.xlabel("Parameters")
  plt.ylabel("Values")
  plt.show()

## Also, interpretations of data.

In [ ]:
# print bar plots for each param based on class

feats = ["Ua","Ui","Ga","Gi","Da","Di","Ba",'Ua/Ui', 'Ua/Ga', 'Ua/Gi', 'Ua/Da', 'Ua/Di', 'Ua/Ba', 'Ga/Gi', 'Ga/Da',
       'Ga/Di', 'Ga/Ba', 'Gi/Da', 'Gi/Di', 'Gi/Ba', 'Da/Di', 'Da/Ba', 'Di/Ba',
       'Ui/Gi', 'Ui/Da', 'Ui/Di', 'Ui/Ba'] + list(feats_df.columns[13:111])

num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster
rows = {}

for i in classes:
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)
  rows[str(i)] = []

colors = ["red","green","blue","purple","orange","turquoise","pink"]

for i in range(len(feats)):
  # Defines what feature we are working with. Dependant on value of i.
  param = feats[i]
  # Outputs the param name
  print(param)
  # Pulls all column names and adds them to param_list
  param_list = cluster_means_df.columns
  # Locates i*4 in param_list
  means = cluster_means_df[param_list[i*4]]
  print(list(means))
  mins = cluster_means_df[param_list[(i*4) + 1]]
  print(list(mins))
  maxes = cluster_means_df[param_list[(i*4) + 2]]
  print(list(maxes))
  stds = cluster_means_df[param_list[(i*4) + 3]]
  print(list(stds))

  # bar plots
  fig, ax = plt.subplots()
  ax.bar(["0","1","2","3","4","5","6"], means, width = 0.3, color = colors[i%6], yerr = stds)



  #set min and max points
  ax.scatter(classes, mins, color = "black")
  ax.scatter(classes, maxes, color = "black")



  ax.set(title = "Mean " + param)

  plt.show()

In [ ]:
# scale each param so we can plot them next to each other
params = ["Ua","Ui","Ga","Gi","Da","Di","Ba"]
# Creates a df excluding all columns but Ua through Ba
params_df = feats_df[params]

from sklearn.preprocessing import MinMaxScaler

feats = []

# picks a parameter
for param in  params:
  # instantiates a MinMaxScaler object
  sc = MinMaxScaler()
  # Reshapes the data and turns it into a column vector and dumps it into a dumpy array. Turns it from a list to a COLUMN
  param_arr = np.array(params_df[param]).reshape(-1,1)
  # Creates a scaled parameter array of the column array (see above)
  param_arr_scaled = sc.fit_transform(param_arr)
  # Creates a new column of the parameters' scaled version
  feats_df[param + "_scaled"] = param_arr_scaled
  #feats.append(param + "_scaled")

# Defines the features
# Should we be hardcoding this?
feats = ['Ua_scaled', 'Ui_scaled', 'Ga_scaled',
        'Gi_scaled', 'Da_scaled', 'Di_scaled', 'Ba_scaled']

feats_extra = []

for i in feats:
  feats_extra.append(i + " mean")
  feats_extra.append(i + " min")
  feats_extra.append(i + " max")
  feats_extra.append(i + " std")

feats = feats_extra
print(len(feats))

num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster
rows = {}

for i in classes:
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)
  rows[str(i)] = []

for i in range(num_classes):
  df = class_dfs[i]
  for j in range(0,len(feats),4):
    mean_val = np.mean(np.array(df[feats[j][0:9]].astype(float)))
    rows[str(i)].append(mean_val)

    min_val = np.min(np.array(df[feats[j][0:9]].astype(float)))
    rows[str(i)].append(min_val)

    max_val = np.max(np.array(df[feats[j][0:9]].astype(float)))
    rows[str(i)].append(max_val)

    std = np.std(np.array(df[feats[j][0:9]].astype(float)))
    rows[str(i)].append(std)

    # print("Cluster " + str(i) + " avg " + feat + ": " + str(mean_val))


cluster_means_df = pd.DataFrame(rows)
# cluster_means_df["feats"] = feats
cluster_means_df = cluster_means_df.transpose()
cluster_means_df.columns = feats

cluster_means_df

In [ ]:
# 6/11/25
# Do all 7 main params on the same plot

# print bar plots for each param based on class

feats = ['Ua_scaled', 'Ui_scaled', 'Ga_scaled',
        'Gi_scaled', 'Da_scaled', 'Di_scaled', 'Ba_scaled']
num_classes = 7
classes = np.arange(0,num_classes)# list of unique clusters

class_dfs = [] # list of dfs for each cluster

for i in classes:
  df = feats_df[feats_df["classifier_pred_class"] == i]
  df.reset_index(inplace = True, drop = True)
  class_dfs.append(df)

colors = ["red","green","blue","purple","orange","turquoise","pink"]
fig, ax = plt.subplots(figsize = (12,8))
for i in range(len(feats)):
  param = feats[i]
  print(param)
  param_list = cluster_means_df.columns
  means = cluster_means_df[param_list[i*4]]
  print(list(means))
  mins = cluster_means_df[param_list[(i*4) + 1]]
  print(list(mins))
  maxes = cluster_means_df[param_list[(i*4) + 2]]
  print(list(maxes))
  stds = cluster_means_df[param_list[(i*4) + 3]]
  print(list(stds))

  # bar plots
  width = 0.32
  # fig, ax = plt.subplots()
  ## ?
  bar_pos = (np.array([0,3,6,9,12,15,18]) + width*i )
  ax.bar(bar_pos, means, width = width, color = colors[i%6], yerr = stds, label = param)



  #set min and max points
  ax.scatter(bar_pos, mins, color = "black")
  ax.scatter(bar_pos, maxes, color = "black")



ax.set(title = "Mean Params By Classes")
ax.legend()

# leg = ax.legend(['Ua_scaled', 'Ui_scaled', 'Ga_scaled',
#         'Gi_scaled', 'Da_scaled', 'Di_scaled', 'Ba_scaled'], loc="upper left")


# leg = ax.get_legend()
# print(type(leg.legend_handles[0]))
# leg.legendHandles[0].set_color('red')
# leg.legendHandles[1].set_color('green')
# leg.legendHandles[2].set_color('blue')
# leg.legendHandles[3].set_color('purple')
# leg.legendHandles[4].set_color('orange')
# leg.legendHandles[5].set_color('turquoise')
# leg.legendHandles[6].set_color('pink')

ax.set_xticks([1,4,7,10,13,16,19])
ax.set_xticklabels(["0",'1', '2', '3', '4', '5', "6"])

plt.show()

In [ ]:
# import pillows library
from PIL import Image, ImageSequence

def show_image(image_num = 4242):
  # define the image path
  path = feats_df["dir"][image_num] + feats_df["path"][image_num]

  # adjust the path for the collab environment
  path_split = path.split("/")
  path_split.insert(2, 'PatternsFormation')
  path_split.insert(3, 'Toybox')
  path = "/".join(path_split)

  # open the image
  img = Image.open(str(path))

  # convert the image into a numpy array
  img = np.array(img)

  #view the image using matplotlib plt.imshow
  plt.imshow(img, cmap = "YlOrRd", vmin = 0, vmax = 255)
  print(path)

show_image()

# print(img[0,199])

In [ ]:
# import pillows library
from PIL import Image, ImageSequence

def show_image(image_num = 4242):
  # define the image path
  path = feats_df["dir"][image_num] + feats_df["path"][image_num]

  # adjust the path for the collab environment
  path_split = path.split("/")
  path_split.insert(2, 'PatternsFormation')
  path_split.insert(3, 'Toybox')
  path = "/".join(path_split)

  # open the image
  img = Image.open(str(path))

  # convert the image into a numpy array
  img = np.array(img)

  #view the image using matplotlib plt.imshow
  plt.imshow(img, cmap = "YlOrRd", vmin = 0, vmax = 255)
  print(path)